In [3]:
import PyPDF2
from transformers import GPT2Tokenizer
from datasets import Dataset
import numpy as np

# Step 1: Read and extract text from the PDF document
def extract_text_from_pdf(pdf_path):
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        text = ""
        for page_num in range(len(reader.pages)):
            page = reader.pages[page_num]
            text += page.extract_text()
    return text

# Step 2: Tokenize the extracted text using the GPT-2 tokenizer
def tokenize_text(text, tokenizer):
    tokens = tokenizer.encode(text)
    return tokens

# Step 3: Split the tokens into segments of 128 tokens each
def split_into_segments(tokens, segment_length=128):
    segments = [tokens[i:i + segment_length] for i in range(0, len(tokens), segment_length) if len(tokens[i:i + segment_length]) == segment_length]

    
    return segments

# Step 4: Save the segments as a Hugging Face datasets.Dataset
def save_as_hf_dataset(segments, output_path):
    segments_int64 = [torch.tensor(segment, dtype=torch.int64) for segment in segments]
    dataset = Dataset.from_dict({"tokens": segments_int64})
    dataset.save_to_disk(output_path)
    return dataset

def main(pdf_path, output_path):
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    
    text = extract_text_from_pdf(pdf_path)
    tokens = tokenize_text(text, tokenizer)
    segments = split_into_segments(tokens)
    
    dataset = save_as_hf_dataset(segments, output_path)
    print(dataset)

# Example usage
pdf_path = "data/blood-m.pdf"
output_path = "data/blood-m"
main(pdf_path, output_path)


Token indices sequence length is longer than the specified maximum sequence length for this model (164821 > 1024). Running this sequence through the model will result in indexing errors
Saving the dataset (1/1 shards): 100%|██████████| 1287/1287 [00:00<00:00, 301618.67 examples/s]

Dataset({
    features: ['tokens'],
    num_rows: 1287
})


In [4]:

from datasets import load_from_disk
ds  = load_from_disk('blood-m')["tokens"]

# print the count of lengths of segments 

for i in range(len(ds)):
    if len(ds[i]) != 128:
        print(len(ds[i]))
